In [1]:
#Day 5, Topic 5: Validating Merges with indicator=True

import pandas as pd

In [3]:
# Create two DataFrames with some overlap
left = pd.DataFrame({
    'ID': [1, 2, 3, 4],
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Score': [85, 90, 78, 92]
})

right = pd.DataFrame({
    'ID': [2, 3, 5, 6],
    'Name': ['Bob', 'Charlie', 'Eve', 'Frank'],
    'Grade': ['A', 'B', 'A', 'C']
})

# Merge with indicator
merged = pd.merge(left, right, on='ID', how='outer', indicator=True)
merged


,ID,Name_x,Score,Name_y,Grade,_merge
0,1,Alice,85.0,NaN,NaN,left_only
1,2,Bob,90.0,Bob,A,both
2,3,Charlie,78.0,Charlie,B,both
3,4,David,92.0,NaN,NaN,left_only
4,5,NaN,NaN,Eve,A,right_only
5,6,NaN,NaN,Frank,C,right_only


In [4]:
#Customizing the Indicator Column Name
merged = pd.merge(left, right, on='ID', how='outer', indicator='source')
merged['source'].value_counts()

source
left_only     2
right_only    2
both          2
Name: count, dtype: int64

In [6]:
#Using indicator with Different Join Types
left_merged = pd.merge(left, right, on='ID', how='left', indicator=True)

left_merged['_merge'].value_counts()

_merge
left_only     2
both          2
right_only    0
Name: count, dtype: int64

In [7]:
#Titanic Example – Validating a Merge
df = pd.read_csv('titanic.csv')
passengers = df[['PassengerId', 'Name', 'Sex', 'Age']].copy()
tickets = df[['PassengerId', 'Pclass', 'Fare', 'Embarked']].copy()
tickets = tickets.drop([2, 5, 10])  # Remove some rows

# Merge with indicator
merged = pd.merge(passengers, tickets, on='PassengerId', how='outer', indicator='Merge_Status')
print(merged['Merge_Status'].value_counts())

Merge_Status
both          888
left_only       3
right_only      0
Name: count, dtype: int64


In [8]:
#Filtering Based on _merge
no_ticket = merged[merged['Merge_Status'] == 'left_only']
no_ticket[['PassengerId', 'Name']]

,PassengerId,Name
2,3,"Heikkinen, Miss. Laina"
5,6,"Moran, Mr. James"
10,11,"Sandstrom, Miss. Marguerite Rut"


In [9]:
#Practice Activity: Validating Merges
import pandas as pd
df = pd.read_csv('titanic.csv')

# Create two DataFrames with intentional mismatches
df_left = df[['PassengerId', 'Name', 'Age']].iloc[:20].copy()  # First 20 passengers
df_right = df[['PassengerId', 'Fare', 'Embarked']].iloc[10:30].copy()  # Passengers 10–29

In [13]:
"""Task 1: Perform an inner join on 'PassengerId' with indicator=True. 
What unique values appear in the _merge column? Why?"""

merged = pd.merge(df_left, df_right, on='PassengerId', how='inner', indicator=True)

print(merged['_merge'].unique())
len(merged)

['both']
Categories (3, object): ['left_only', 'right_only', 'both']


10

In [14]:
"""Task 2: Perform a left join on 'PassengerId' with indicator=True. 
Count how many rows have each merge status ('both' vs. 'left_only')."""

merged = pd.merge(df_left, df_right, on='PassengerId', how='left', indicator=True)

merged['_merge'].value_counts()

_merge
left_only     10
both          10
right_only     0
Name: count, dtype: int64

In [15]:
"""Task 3: Perform a full outer join on 'PassengerId' with indicator='Source'. 
Display the Source value counts. How many passengers are in both, left only, and right only?"""

merged = pd.merge(df_left, df_right, on='PassengerId', how='outer', indicator='Source')
merged['Source'].value_counts()

Source
left_only     10
right_only    10
both          10
Name: count, dtype: int64

In [16]:
"""Task 4 (Bonus): Use the result from Task 3 to extract only the rows that exist in 
df_right but NOT in df_left. Display their PassengerId and Fare."""

merged = pd.merge(df_left, df_right, on='PassengerId', how='outer', indicator=True)
result = merged[merged['_merge'] == 'right_only']

result[['PassengerId', 'Fare']]

,PassengerId,Fare
20,21,26.0000
21,22,13.0000
22,23,8.0292
23,24,35.5000
24,25,21.0750
25,26,31.3875
26,27,7.2250
27,28,263.0000
28,29,7.8792
29,30,7.8958


In [18]:
#Day 5, Topic 6: Combining Overlapping DataFrames with combine_first()

In [19]:
import pandas as pd
import numpy as np
# Primary DataFrame (has some missing values)
df1 = pd.DataFrame({
    'A': [1, np.nan, 3, np.nan],
    'B': [np.nan, 20, np.nan, 40]
})

# Backup DataFrame (has values where df1 is missing)
df2 = pd.DataFrame({
    'A': [10, 20, 30, 40],
    'B': [100, 200, 300, 400]
})
result = df1.combine_first(df2)
result

,A,B
0,1.0,100.0
1,20.0,20.0
2,3.0,300.0
3,40.0,40.0


In [20]:
#Partial Overlap – Aligns on Both Index and Columns

df1 = pd.DataFrame({'A': [1, np.nan]}, index=['r1', 'r2'])
df2 = pd.DataFrame({'A': [10, 20]}, index=['r1', 'r2'])

result = df1.combine_first(df2)
result

,A
r1,1.0
r2,20.0


In [21]:
#Titanic Use Case – Filling Missing Ages
df = pd.read_csv('titanic.csv')

df['Age_Estimated'] = df.groupby('Pclass')['Age'].transform('median')
df['Age_Final'] = df['Age'].combine_first(df['Age_Estimated'])

df[['Age', 'Age_Estimated', 'Age_Final']].isna().sum()

Age              177
Age_Estimated      0
Age_Final          0
dtype: int64

In [22]:
#🛠️ Practice Activity: combine_first()
import pandas as pd
import numpy as np

# Primary data (has missing values)
df_main = pd.DataFrame({
    'PassengerId': [1, 2, 3, 4, 5],
    'Age': [22, np.nan, 26, np.nan, 35],
    'Fare': [7.25, 71.28, np.nan, 53.10, 8.05]
}).set_index('PassengerId')

# Backup data (has some overlapping and some new rows/columns)
df_backup = pd.DataFrame({
    'PassengerId': [2, 4, 6],
    'Age': [30, 28, 40],
    'Fare': [70.0, 55.0, 10.0],
    'Embarked': ['S', 'C', 'Q']
}).set_index('PassengerId')

In [23]:
"""Task 1: Use combine_first() to fill missing values in df_main with values from df_backup. 
Display the result. What happens to the 'Embarked' column?
"""

result = df_main.combine_first(df_backup)

result

,Age,Embarked,Fare
PassengerId,,,
1,22.0,NaN,7.25
2,30.0,S,71.28
3,26.0,NaN,NaN
4,28.0,C,53.10
5,35.0,NaN,8.05
6,40.0,Q,10.00


In [24]:
print("\n=== Task 2: PassengerId = 6 ===")
print("Is PassengerId 6 in the result?", 6 in result.index)
print("Value for PassengerId 6:\n", result.loc[6])
print("Why: combine_first() adds rows from df_backup that are missing in df_main.")


=== Task 2: PassengerId = 6 ===
Is PassengerId 6 in the result? True
Value for PassengerId 6:
 Age         40.0
Embarked       Q
Fare        10.0
Name: 6, dtype: object
Why: combine_first() adds rows from df_backup that are missing in df_main.


In [25]:
print("\n=== Task 3: fillna() vs combine_first() ===")
fillna_result = df_main.fillna(df_backup)
print("fillna() result:\n", fillna_result)
print("\nDifferences:")
print("1. fillna() does NOT add the 'Embarked' column.")
print("2. fillna() does NOT add the new row (PassengerId = 6).")


=== Task 3: fillna() vs combine_first() ===
fillna() result:
               Age   Fare
PassengerId             
1            22.0   7.25
2            30.0  71.28
3            26.0    NaN
4            28.0  53.10
5            35.0   8.05

Differences:
1. fillna() does NOT add the 'Embarked' column.
2. fillna() does NOT add the new row (PassengerId = 6).


In [26]:
print("\n=== Task 4 (Bonus): Fare_Filled using fillna() ===")
df_main['Fare_Filled'] = df_main['Fare'].fillna(df_backup['Fare'])
print(df_main[['Fare', 'Fare_Filled']])
print("\nObservation: Missing Fare for PassengerId 3 is filled with 70.0 from df_backup.")
print("No new rows or columns are added.")


=== Task 4 (Bonus): Fare_Filled using fillna() ===
              Fare  Fare_Filled
PassengerId                    
1             7.25         7.25
2            71.28        71.28
3              NaN          NaN
4            53.10        53.10
5             8.05         8.05

Observation: Missing Fare for PassengerId 3 is filled with 70.0 from df_backup.
No new rows or columns are added.
